# MOSS-Image-SFT on Kaggle T4 via ComfyUI (Option A)

記事 https://ai-hardware-zukan.com/moss-image-vram-16gb/ の構成を Kaggle T4 (16GB VRAM) で再現する。

- ComfyUI v0.42.1（記事と同じバージョン、MOSS-Image 対応）
- 量子化モデル3点と 4ステップ LoRA は Kaggle Dataset `masashiki/moss-image-sft-quantized` からマウントし、`/comfyui/models/` へ**コピー**（読み取り専用マウント経由の推論は極端に遅いため。HF からの再DLは不要。マウントが無い場合は HF からDLするフォールバックあり）:
  - DiT SFT int8: `diffusion_models/moss_image_sft_int8_convrot.safetensors` (5.04 GiB)
  - TE w4a8 GPTQ: `text_encoders/qwen3.5_9b_w4a8_gptq.safetensors` (6.84 GiB)
  - VAE small: `vae/flux2-vae_small.safetensors` (0.23 GiB)
  - 4ステップ LoRA: `loras/moss_image_sft_4_steps_lora_v2_600_avg_rank_72_bf16.safetensors` (約0.33 GB、bf16)
- テンプレート `image_moss_image_t2i` ベースだが、**4ステップ蒸留 LoRA**（`Kijai/MOSS-image_experimental` の `moss_image_sft_4_steps_lora_v2_600_avg_rank_72_bf16.safetensors`、bf16）を `LoraLoaderModelOnly`（strength 1.0）で適用し、**steps=4 / CFG 1.0 / euler / simple**（公式 ComfyUI ワークフロー `moss-image-4step-turbo.json` の設定）で生成する
- ベース DiT は int8、LoRA は bf16 という組み合わせは未検証。T4 は bf16 非ネイティブなので、dtype 変換の挙動と生成時間をログで確認する
- プロンプト拡張（TextGenerate ノード）は T4 では遅いので**無効**。生プロンプトをそのまま使う
- モデルは `/comfyui/models/` に置く（`/kaggle/working` は 20GB しかないので使わない）
- ComfyUI は headless で起動し、HTTP API (`/prompt`) でワークフローを実行する
- 生成画像は `/kaggle/working/` に保存される（`kaggle kernels output -o` で取得可能）
- LoRA の dtype は safetensors ヘッダから直接読み取って表示する
- 生成後に ComfyUI のログから lora/dtype/bf16/error を含む行を抽出して表示する


In [ ]:
import shutil, subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
for p in ["/", "/kaggle/working"]:
    u = shutil.disk_usage(p)
    print(p, f"free={u.free/1e9:.1f}GB")

In [ ]:
import os, subprocess
if not os.path.exists("/comfyui"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "v0.42.1",
                    "https://github.com/comfyanonymous/ComfyUI.git", "/comfyui"], check=True)
print(subprocess.run(["git", "-C", "/comfyui", "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/comfyui/requirements.txt"], check=True)
print("requirements installed")

In [ ]:
import os
print("/kaggle/input exists:", os.path.exists("/kaggle/input"))
if os.path.exists("/kaggle/input"):
    print("/kaggle/input:", os.listdir("/kaggle/input"))
    for root, dirs, files in os.walk("/kaggle/input"):
        print(root, "dirs:", dirs, "files:", files[:5])
        if root.count("/") >= 4:
            break
CANDIDATES = [
    "/kaggle/input/moss-image-sft-quantized",
    "/kaggle/input/datasets/masashiki/moss-image-sft-quantized",
]
SRC = next((p for p in CANDIDATES if os.path.isdir(p)), None)
print("dataset mount:", SRC)
if SRC:
    import shutil, time
    for sub in ["diffusion_models", "text_encoders", "vae", "loras"]:
        os.makedirs(f"/comfyui/models/{sub}", exist_ok=True)
        for f in os.listdir(f"{SRC}/{sub}"):
            dst = f"/comfyui/models/{sub}/{f}"
            if not os.path.exists(dst):
                t0 = time.time()
                shutil.copy2(f"{SRC}/{sub}/{f}", dst)
                print(f"copied: {dst} ({time.time()-t0:.1f}s)")
else:
    print("dataset mount not found, falling back to HF download")
    from huggingface_hub import hf_hub_download
    files = [
        ("Comfy-Org/MOSS-Image", "diffusion_models/moss_image_sft_int8_convrot.safetensors"),
        ("Comfy-Org/MOSS-Image", "text_encoders/qwen3.5_9b_w4a8_gptq.safetensors"),
        ("Comfy-Org/MOSS-Image", "vae/flux2-vae_small.safetensors"),
        ("Kijai/MOSS-image_experimental", "loras/moss_image_sft_4_steps_lora_v2_600_avg_rank_72_bf16.safetensors"),
    ]
    for repo, fname in files:
        p = hf_hub_download(repo_id=repo, filename=fname, local_dir="/comfyui/models")
        print("downloaded:", p)

lora_path = "/comfyui/models/loras/moss_image_sft_4_steps_lora_v2_600_avg_rank_72_bf16.safetensors"

import collections
from safetensors import safe_open
with safe_open(lora_path, framework="pt") as f:
    keys = list(f.keys())
    dtypes = collections.Counter(f.get_slice(k).get_dtype() for k in keys)
    print("num tensors:", len(keys))
    print("dtypes:", dict(dtypes))
    print("sample keys:", keys[:5])

In [ ]:
import subprocess, time, urllib.request
log = open("/comfyui.log", "w")
proc = subprocess.Popen(["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"],
                        cwd="/comfyui", stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
print("pid:", proc.pid)
ready = False
for i in range(300):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2) as r:
            print("ComfyUI ready after", i, "s")
            ready = True
            break
    except Exception:
        time.sleep(1)
if not ready:
    print(open("/comfyui.log").read()[-4000:])
    raise RuntimeError("ComfyUI did not start in 300s")

In [ ]:
import json, time, urllib.request, uuid, shutil, os

PROMPT = "Cinematic night street photograph: a model in an exaggerated balloon-sleeve white wool coat walking toward camera under a huge illuminated theatre marquee, the marquee bulbs spelling \"MOSS\" in tall capital letters, wet asphalt with long reflections, warm tungsten glow against cold blue night, 35mm film grain, low angle"
NEGATIVE = ""
SEED = 42
STEPS = 4
CFG = 1.0
WIDTH, HEIGHT = 1024, 1024

workflow = {
    "1": {"class_type": "UNETLoader", "inputs": {"unet_name": "moss_image_sft_int8_convrot.safetensors", "weight_dtype": "default"}},
    "10": {"class_type": "LoraLoaderModelOnly", "inputs": {"model": ["1", 0], "lora_name": "moss_image_sft_4_steps_lora_v2_600_avg_rank_72_bf16.safetensors", "strength_model": 1.0}},
    "2": {"class_type": "CLIPLoader", "inputs": {"clip_name": "qwen3.5_9b_w4a8_gptq.safetensors", "type": "moss_image", "device": "default"}},
    "3": {"class_type": "VAELoader", "inputs": {"vae_name": "flux2-vae_small.safetensors"}},
    "4": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["2", 0], "text": PROMPT}},
    "5": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["2", 0], "text": NEGATIVE}},
    "6": {"class_type": "EmptyFlux2LatentImage", "inputs": {"width": WIDTH, "height": HEIGHT, "batch_size": 1}},
    "7": {"class_type": "KSampler", "inputs": {"model": ["10", 0], "positive": ["4", 0], "negative": ["5", 0],
                                        "latent_image": ["6", 0], "seed": SEED, "steps": STEPS, "cfg": CFG,
                                        "sampler_name": "euler", "scheduler": "simple", "denoise": 1}},
    "8": {"class_type": "VAEDecode", "inputs": {"samples": ["7", 0], "vae": ["3", 0]}},
    "9": {"class_type": "SaveImage", "inputs": {"images": ["8", 0], "filename_prefix": "moss_image_4step"}},
}

req = urllib.request.Request("http://127.0.0.1:8188/prompt",
                             data=json.dumps({"prompt": workflow, "client_id": str(uuid.uuid4())}).encode(),
                             headers={"Content-Type": "application/json"})
with urllib.request.urlopen(req) as r:
    resp = json.loads(r.read())
print(resp)
prompt_id = resp["prompt_id"]

t0 = time.time()
while True:
    with urllib.request.urlopen(f"http://127.0.0.1:8188/history/{prompt_id}") as r:
        hist = json.loads(r.read())
    if prompt_id in hist:
        break
    if time.time() - t0 > 1800:
        raise TimeoutError("generation timed out after 1800s")
    time.sleep(2)
elapsed = time.time() - t0
print(f"generation took {elapsed:.1f}s")

for node_id, out in hist[prompt_id]["outputs"].items():
    for img in out.get("images", []):
        sub = img.get("subfolder") or ""
        src = os.path.join("/comfyui/output", sub, img["filename"])
        dst = os.path.join("/kaggle/working", img["filename"])
        shutil.copy(src, dst)
        print("saved:", dst)

print("=== ComfyUI log (lora/dtype/bf16/error lines) ===")
shown = 0
for line in open("/comfyui.log", errors="ignore"):
    if any(kw in line for kw in ["lora", "LoRA", "dtype", "bf16", "Error", "error", "Traceback"]):
        print(line.rstrip())
        shown += 1
        if shown >= 80:
            print("... (truncated)")
            break

In [ ]:
import glob
from PIL import Image
for p in sorted(glob.glob("/kaggle/working/*.png")):
    im = Image.open(p)
    print(p, im.size)
    display(im)